<a href="https://colab.research.google.com/github/EmmanuelC-137/Mineria_de_datos/blob/main/Proyecto_mineria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Proyecto de Mineria de datos

# Proyecto Minería de Datos: Reconocimiento de Voz "Luz Inteligente"

**Integrantes:**
- Martin Martinez Zuñiga
- Juan Antonio Cabrera Meza
- Emmanuel Hernández Méndez

**Grupo(s):** [6:00–7:00 PM / 8:00–9:00 PM]

**Fecha de entrega:** 25 de mayo de 2026

---

## Índice
1. [Fase 1: Preparación de la Libreta Colab](#fase1)
2. [Fase 2: Adquisición y Procesamiento de Datos](#fase2)
3. [Fase 3: Desarrollo del Modelo y Optimización](#fase3)
4. [Fase 4: Evaluación y Resultados](#fase4)
5. [Conclusiones](#conclusiones)

---

## Introducción Teórica

### Procesamiento de audio
El procesamiento de audio implica la manipulación y análisis de señales acústicas. En el contexto del aprendizaje automático, el audio crudo (una secuencia de amplitudes en el tiempo) es difícil de procesar directamente debido a su alta dimensionalidad y variabilidad. Por ello, se recurre a técnicas de extracción de características.

### Uso de MFCC (Coeficientes Cepstrales en Frecuencia Mel)
Los MFCC son representaciones de características ampliamente utilizadas en el procesamiento del habla. Se basan en la percepción humana del sonido, utilizando la escala Mel, que aproxima la respuesta no lineal del oído humano a diferentes frecuencias. Los MFCC capturan la envolvente espectral de un sonido, lo cual es fundamental para distinguir diferentes fonemas y palabras.

### Redes Neuronales Densas (Perceptrón Multicapa)
Las redes neuronales densas, o completamente conectadas, son arquitecturas donde cada neurona de una capa está conectada a todas las neuronas de la capa siguiente. Son útiles para aprender relaciones complejas no lineales en datos tabulares o características extraídas (como los MFCC aplanados).

<a id='fase1'></a>
## Fase 1: Preparación de la Libreta Colab

### Paso 1.2: Importación de herramientas
Instalamos e importamos las librerías necesarias para el procesamiento de audio y construcción del modelo.

In [ ]:
import os
import librosa
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
import matplotlib.pyplot as plt

<a id='fase2'></a>
## Fase 2: Adquisición y Procesamiento de Datos

### Paso 2.1: Recolección del dataset
Montamos Google Drive para acceder a la carpeta donde están guardados los audios (122 archivos esperados).

In [1]:
# Montar Google Drive (útil si usas Colab)
from google.colab import drive
drive.mount('/content/drive')

# NOTA: Cambia esta ruta a la ubicación de tus audios en Drive (o ruta local)
dataset_path = '/content/drive/MyDrive/Librosa'

Mounted at /content/drive


### Paso 2.2 y 2.3: Extracción de características (MFCC), Normalización y Padding
Definiremos una función para extraer los MFCC de cada audio, asegurando que todos tengan el mismo tamaño (padding) y normalizando los valores.

In [ ]:
def extract_features(file_path, max_pad_len=44):
    try:
        # Cargar audio
        audio, sample_rate = librosa.load(file_path, res_type='kaiser_fast')
        # Extraer MFCC
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)

        # Padding o truncamiento para asegurar el mismo tamaño
        pad_width = max_pad_len - mfccs.shape[1]
        if pad_width > 0:
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        # Normalización (estandarización a media 0 y varianza 1 para ayudar al modelo)
        mfccs_normalized = (mfccs - np.mean(mfccs)) / np.std(mfccs)

        return mfccs_normalized
    except Exception as e:
        print(f"Error procesando {file_path}: {e}")
        return None

# Aquí deberás iterar sobre las carpetas de tu dataset para extraer las características (X)
# y asignar las etiquetas correspondientes (y).
# Por ejemplo:
# 0 para "Luz Apagada"
# 1 para "Luz Encendida"

# Ejemplo de estructura de código a completar:
# X = []
# y = []
# ... lógica para leer archivos ...
# X = np.array(X)
# y = np.array(y)

<a id='fase3'></a>
## Fase 3: Desarrollo del Modelo y Optimización

### Paso 3.1 y 3.2: Arquitectura base y Experimentación (Tuning)
Crearemos una función que nos permita probar diferentes arquitecturas variando el número de capas ocultas y neuronas.

**Justificación de Arquitectura Elegida:**
*(Espacio para que el equipo justifique la arquitectura elegida tras las pruebas, número de neuronas, ajustes, etc.)*

In [ ]:
def create_model(num_hidden_layers, neurons_per_layer, input_shape):
    model = Sequential()
    # Capa de Entrada (flatten)
    model.add(Flatten(input_shape=input_shape))

    # Capas ocultas (densas)
    for _ in range(num_hidden_layers):
        model.add(Dense(neurons_per_layer, activation='relu'))
        # Dropout opcional para evitar sobreajuste
        # model.add(Dropout(0.2))

    # Capa de salida con activación sigmoide para clasificación binaria
    model.add(Dense(1, activation='sigmoid'))

    # Compilar el modelo
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# Ejemplo de input_shape basado en nuestros MFCCs (40, max_pad_len)
# input_shape = (40, 44)
# model = create_model(num_hidden_layers=2, neurons_per_layer=64, input_shape=input_shape)
# model.summary()

### Paso 3.3: Entrenamiento
Entrenamos el modelo y graficamos la pérdida (Loss) y la precisión (Accuracy).

In [ ]:
# Entrenamiento (descomenta y usa tus datos X_train, y_train, X_val, y_val)
# history = model.fit(X_train, y_train, epochs=30, batch_size=8, validation_data=(X_val, y_val))

def plot_history(history):
    plt.figure(figsize=(12, 4))

    # Gráfica de Loss (pérdida)
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Pérdida del Modelo')
    plt.xlabel('Época')
    plt.ylabel('Loss')
    plt.legend()

    # Gráfica de Accuracy (precisión)
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    if 'val_accuracy' in history.history:
        plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Precisión del Modelo')
    plt.xlabel('Época')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.show()

# plot_history(history)

<a id='fase4'></a>
## Fase 4: Evaluación y Resultados

### Paso 4.1: Simulación de salida
Implementamos la lógica para interpretar la predicción del modelo.

In [ ]:
def interpret_prediction(prediction):
    if prediction > 0.5:
        return "Luz Encendida"
    else:
        return "Luz Apagada"

# Ejemplo de uso con datos de prueba:
# prob = model.predict(X_test[0:1])
# print(f"Resultado: {interpret_prediction(prob[0][0])}")

### Paso 4.2: Prueba ciega (voz del profesor)
Cargaremos un audio externo, aplicaremos el mismo preprocesamiento y evaluaremos el resultado para ver si el modelo generaliza correctamente.

In [ ]:
def predict_audio(file_path, model):
    # 1. Aplicar el mismo preprocesamiento
    features = extract_features(file_path)
    if features is not None:
        # Añadir la dimensión del batch
        features = np.expand_dims(features, axis=0)
        # 2. Predecir
        prediction = model.predict(features)
        # 3. Interpretar
        result = interpret_prediction(prediction[0][0])
        print(f"Predicción para el audio '{file_path}':")
        print(f" -> {result} (Probabilidad de encendido: {prediction[0][0]:.4f})")
    else:
        print("No se pudo procesar el audio para la predicción.")

# path_audio_profesor = '/content/drive/MyDrive/Ruta/A/Prueba/audio_profesor.wav'
# predict_audio(path_audio_profesor, model)

<a id='conclusiones'></a>
## Conclusiones
